# 🔬 Virtual Lab 4: Running Llama Models and Deepseek on Replicate  

<div style="border: 2px solid #4CAF50; padding: 15px; border-radius: 10px; background-color: #f4f4f4;">

### 🚀 **Platform**  
**Replicate**  

### 🏷️ **Models Used**  
- **meta/meta-llama-3-70b-instruct**  
- **meta-llama-3.1-405b-instruct**  
- **meta/meta-llama-3-8b-instruct**  
- **deepseek-ai/deepseek-r1**  

### 🛠️ **Frameworks Used**  
- **LlamaIndex**  
- **LangChain / LangGraph**  

</div>

## Installation and Setup

In [ ]:
!pip install llama-index
!pip install llama-index-llms-ollama
!pip install llama-index-llms-replicate
!pip install llama-index-embeddings-huggingface
!pip install llama-parse
!pip install replicate

In [1]:
import nest_asyncio

nest_asyncio.apply()

### Setup LLM using Replicate

Make sure you have REPLICATE_API_TOKEN specified!

In [1]:
import os
import replicate

# Set the REPLICATE_API_TOKEN environment variable
# os.environ['REPLICATE_API_TOKEN'] = ''

In [2]:
from llama_index.llms.replicate import Replicate

llm_replicate = Replicate(model='deepseek-ai/deepseek-r1', api_token=os.environ['REPLICATE_API_TOKEN'])
#llm_replicate = Replicate(model='meta/meta-llama-3-70b-instruct', api_token=os.environ['REPLICATE_API_TOKEN'])

### Setup Embedding Model

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

### Define Global Settings Configuration

In LlamaIndex, you can define global settings so you don't have to pass the LLM / embedding model objects everywhere.

In [4]:
from llama_index.core import Settings

Settings.llm = llm_replicate
Settings.embed_model = embed_model

### Download Data

Here you'll download data that's used in section 2 and onwards.

We'll download some articles on Kendrick, Drake, and their beef (as of May 2024).

In [5]:
!mkdir -p data
!curl -L 'https://www.dropbox.com/scl/fi/t1soxfjdp0v44an6sdymd/drake_kendrick_beef.pdf?rlkey=u9546ymb7fj8lk2v64r6p5r5k&st=wjzzrgil&dl=1' -o data/drake_kendrick_beef.pdf
!curl -L 'https://www.dropbox.com/scl/fi/nts3n64s6kymner2jppd6/drake.pdf?rlkey=hksirpqwzlzqoejn55zemk6ld&st=mohyfyh4&dl=1' -o data/drake.pdf
!curl -L 'https://www.dropbox.com/scl/fi/8ax2vnoebhmy44bes2n1d/kendrick.pdf?rlkey=fhxvn94t5amdqcv9vshifd3hj&st=dxdtytn6&dl=1' -o data/kendrick.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     48      0 --:--:-- --:--:-- --:--:--    48
100   475    0   475    0     0    621      0 --:--:-- --:--:-- --:--:--   621
100 47.0M  100 47.0M    0     0  7031k      0  0:00:06  0:00:06 --:--:-- 8264k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     51      0 --:--:-- --:--:-- --:--:--    51
100   475    0   475    0     0    663      0 --:--:-- --:--:-- --:--:--   663
100 4483k  100 4483k    0     0  2620k      0  0:00:01  0:00:01 --:--:-- 5406k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     38      0 --:--:

### Load Data

We load data using LlamaParse by default, but you can also choose to opt for our free pypdf reader (in SimpleDirectoryReader by default) if you don't have an account! 

1. LlamaParse: Signup for an account here: cloud.llamaindex.ai. You get 1k free pages a day, and paid plan is 7k free pages + 0.3c per additional page. LlamaParse is a good option if you want to parse complex documents, like PDFs with charts, tables, and more. 

2. Default PDF Parser (In `SimpleDirectoryReader`). If you don't want to signup for an account / use a PDF service, just use the default PyPDF reader bundled in our file loader. It's a good choice for getting started!

In [6]:
from llama_index.core import SimpleDirectoryReader

docs_kendrick = SimpleDirectoryReader(input_files=['data/kendrick.pdf']).load_data()
docs_drake = SimpleDirectoryReader(input_files=['data/drake.pdf']).load_data()
docs_both = SimpleDirectoryReader(input_files=['data/drake_kendrick_beef.pdf']).load_data()

## 1. Basic Completion and Chat

### Call complete with a prompt

In [7]:
response = llm_replicate.complete('do you like drake or kendrick better?')

print(response)

<think>
Okay, the user is asking if I prefer Drake or Kendrick Lamar. First, I need to consider what each artist represents. Drake is hugely popular with a more mainstream, versatile style, blending rap with R&B and pop elements. His hits are catchy and he's known for his emotive lyrics. On the other hand, Kendrick is critically acclaimed, often praised for his storytelling, social commentary, and complex lyrics. Albums like "To Pimp a Butterfly" are considered masterpieces.

Now, the user might be looking for a recommendation or just curious about my opinion. But as an AI, I don't have personal preferences. However, I can analyze their styles and impact. I should explain the strengths of each artist without bias. Maybe mention their different approaches to music: Drake's versatility and chart success versus Kendrick's depth and critical acclaim. Also, consider their fan bases and cultural impact. The user might be a fan of one and looking to validate their preference, or trying to dec

In [8]:
stream_response = \
    llm_replicate.stream_complete("you're a drake fan. tell me why you like drake more than kendrick")

for t in stream_response:
    print(t.delta, end='')

<think>
Okay, so the user wants me to explain why I, as a Drake fan, like Drake more than Kendrick Lamar. Let me start by recalling the main points people usually bring up when comparing these two. Drake is known for his versatility in music—he does rap, R&B, pop, and even dancehall. His melodies are catchy, and he's had a ton of commercial success. On the other hand, Kendrick is often praised for his lyrical depth, storytelling, and social commentary.

Hmm, so as a Drake fan, I need to highlight the aspects that make Drake stand out compared to Kendrick. Maybe start with his versatility? Like, Drake can switch between different genres smoothly. Tracks like "Hold On, We're Going Home" versus "God's Plan" show his range. Also, his emotional vulnerability in songs about relationships, insecurities, which makes him relatable.

Then there's the consistency and work ethic. Drake drops music frequently and stays relevant. Kendrick takes longer between albums, which is fine for quality, but D

### Call chat with a list of messages

In [9]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role='system', content='You are Kendrick.'),
    ChatMessage(role='user', content='Write a verse.'),
]
response = llm_replicate.chat(messages)

In [10]:
print(response)

assistant: <think>
Alright, let's tackle this. The user wants me to write a verse as Kendrick Lamar. So first, I need to channel his style. Kendrick is known for his intricate wordplay, storytelling, and social commentary. His verses often have a strong narrative, use internal rhymes, and metaphors.

I should start by brainstorming themes. Common themes in his music include struggle, identity, systemic issues, and personal growth. Maybe touch on something like societal pressures or personal triumph. Let's go with societal issues, since that's a recurring theme.

Next, structure. Kendrick often uses AABB or ABAB rhyme schemes, but he's also known for switching flows. I need to make sure the verse has a good rhythm, maybe some alliteration and assonance. Let me think of a metaphor. Maybe "crown of thorns" to symbolize the burdens of leadership or societal expectations.

First line: "Crown of thorns, but the kingdom’s on ashes," sets a regal yet fallen image. Then follow up with internal 

## 2. Basic RAG (Vector Search, Summarization)

### Basic RAG (Vector Search)

In [11]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(docs_both)
query_engine = index.as_query_engine(similarity_top_k=3)

In [12]:
response = query_engine.query('Tell me about family matters')

In [13]:
print(str(response))

<think>
Alright, let's try to figure out how to answer the query "Tell me about family matters" using the given context. First, I need to understand what "Family Matters" refers to here. From the context, it looks like it's a diss track by Drake directed at Kendrick Lamar as part of their beef.

The user wants information about "Family Matters," so I need to extract all relevant details from the provided pages. Let me go through each page mentioned: 21, 24, 29.

Starting with page 21: It mentions that "Family Matters" is a seven-and-a-half-minute diss track with an accompanying video. Drake uses three different beats, making it like three songs in one. He addresses not just Kendrick but also other opponents like Metro Boomin, Rick Ross, The Weeknd, and A$AP Rocky. The non-Kendrick bars are described as distracting and some of Drake's laziest writing. He accuses Kendrick of various things like infidelity, not spending time with his family, and domestic abuse. There's also a mention of D

### Basic RAG (Summarization)

In [14]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex.from_documents(docs_both)
summary_engine = summary_index.as_query_engine()

In [15]:
response = \
    summary_engine.query("Given your assessment of this article, who won the beef?")

In [16]:
print(str(response))

<think>
Okay, let's parse this new context. The user provided additional pages from the GQ article, but most of them seem to be unrelated content like sponsored links, newsletter sign-ups, or other articles (e.g., about Shaboozey, A$AP Rocky, etc.). The only relevant snippet is on page 30, where Drake's "Family Matters" is discussed. The key point here is that Drake tries to preempt Kendrick's response by joking that Kendrick has "10 more records to drop," anticipating a prolonged battle.

This adds a bit more context to Drake's strategy in "Family Matters." The original refinement already mentioned Drake's "Family Matters" as a weaker response, but this snippet shows Drake attempting to undermine Kendrick's momentum by suggesting Kendrick would keep releasing tracks. However, as the original answer noted, Kendrick did drop multiple tracks quickly ("Meet the Grahams" and "Not Like Us"), which overwhelmed Drake's efforts. 

So, integrating this detail would strengthen the point about Ke

## 3. Advanced RAG (Routing, Sub-Questions)

### Build a Router that can choose whether to do vector search or summarization

In [17]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

vector_tool = \
    QueryEngineTool(
        index.as_query_engine(),
        metadata= \
            ToolMetadata(
                name='vector_search',
                description='Useful for searching for specific facts.',
            ),
    )
summary_tool = \
    QueryEngineTool(
        index.as_query_engine(response_mode='tree_summarize'),
        metadata= \
            ToolMetadata(
                name='summary',
                description='Useful for summarizing an entire document.',
            ),
    )

In [18]:
from llama_index.core.query_engine import RouterQueryEngine

query_engine = \
    RouterQueryEngine.from_defaults(
        [vector_tool, summary_tool], 
        select_multi=False,
        verbose=True,
    )

response = \
    query_engine.query('Tell me about the song meet the grahams - why is it significant')

Selecting query engine 0: The question asks for specific significance of the song 'Meet the Grahams,' which requires locating factual details about its context, themes, or cultural impact rather than summarizing a full document..


In [19]:
print(response)

<think>
Okay, so I need to figure out why "Meet the Grahams" is significant based on the given context. Let me look through the provided information.

First, in the context from page 26, it mentions that after "Meet the Grahams" impacted, Drake laughed it off as a sign that team PG is desperate and in shambles. Then Kendrick released "Not Like Us," which shifted the momentum. Also, "Meet the Grahams" is described as a more uncomfortable and joyless track compared to others in the beef. The article contrasts it with "Not Like Us," which is more of a block party vibe. 

On page 27, there's a mention that "Meet the Grahams" was a sign of desperation according to Drake, but Kendrick followed up with "Not Like Us" to prove he's more war-ready. Also, "Meet the Grahams" is noted for its serious and personal attacks, lacking the levity that "Not Like Us" has. The track is part of the escalation where Kendrick is gathering support in the culture, using more intense and personal disses.

Putting

### Break Complex Questions down into Sub-Questions

Our Sub-Question Query Engine breaks complex questions down into sub-questions.


In [20]:
drake_index = VectorStoreIndex.from_documents(docs_drake)
drake_query_engine = drake_index.as_query_engine(similarity_top_k=3)

kendrick_index = VectorStoreIndex.from_documents(docs_kendrick)
kendrick_query_engine = kendrick_index.as_query_engine(similarity_top_k=3)

In [21]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

drake_tool = \
    QueryEngineTool(
        drake_index.as_query_engine(),
        metadata= \
            ToolMetadata(
                name='drake_search',
                description="Useful for searching over Drake's life.",
            ),
    )
kendrick_tool = \
    QueryEngineTool(
        kendrick_index.as_query_engine(),
        metadata= \
            ToolMetadata(
                name='kendrick_summary',
                description="Useful for searching over Kendrick's life.",
            ),
    )

In [22]:
from llama_index.core.query_engine import SubQuestionQueryEngine

query_engine = \
    SubQuestionQueryEngine.from_defaults(
        [drake_tool, kendrick_tool],
        llm=llm_replicate,  
        verbose=True,
    )

response = query_engine.query('Which albums did Drake release in his career?')

print(response)

Generated 1 sub questions.
[drake_search] Q: Which albums did Drake release in his career?
[drake_search] A: <think>
Okay, let me tackle this question about Drake's albums. The user wants to know which albums he released in his career, based solely on the provided context.

First, I'll scan through the context to find mentions of his albums. The text mentions a few key sections. Starting on page 6, there's a section labeled "2013–2015: Nothing Was the Same and If You're Reading This It's Too Late". That right there gives two album titles: "Nothing Was the Same" and "If You're Reading This It's Too Late". 

Then, right after that, there's another heading: "2015–2017: What a Time to Be Alive, Views, and More Life". This lists three more projects: "What a Time to Be Alive", "Views", and "More Life". However, I need to check if these are all considered albums. The text refers to "What a Time to Be Alive" as a collaborative mixtape with Future, so maybe that's a mixtape rather than an album

## 4. Text-to-SQL 

Here, we download and use a sample SQLite database with 11 tables, with various info about music, playlists, and customers. We will limit to a select few tables for this test.

In [23]:
# Function to download files
def download_file(url, filepath):
    response = requests.get(url, stream=True)
    with open(filepath, 'wb') as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)

# Create data directory
os.makedirs('data', exist_ok=True)

In [24]:
# Download and extract SQLite database
import requests
import zipfile
download_file('https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip', 'data/chinook.zip')
with zipfile.ZipFile('data/chinook.zip', 'r') as zip_ref:
    zip_ref.extractall('data/')

In [25]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    String,
    Integer,
    select,
    column,
)

engine = create_engine('sqlite:///data/chinook.db')

In [26]:
from llama_index.core import SQLDatabase

sql_database = SQLDatabase(engine)

In [27]:
from llama_index.core.indices.struct_store import NLSQLTableQueryEngine

query_engine = \
    NLSQLTableQueryEngine(
        sql_database=sql_database,
        tables=['albums', 'tracks', 'artists'],
        llm=llm_replicate,
    )

In [28]:
response = query_engine.query('What are some albums?')

print(response)

<think>
Okay, the user asked for some albums, and I need to present the SQL query results in a clear way. Let's look at the data from the SQL response. The first album is "...And Justice For All" by Metallica. That's a classic. Then there's the Scorpions' best of collection. The third entry is Aaron Copland with an orchestra, which seems like a classical album. Next are two Iron Maiden albums. 

I should list these out but make sure to mention both the album titles and the artists. Maybe start with a sentence introducing the list. Since there are five entries, I should number them or use bullet points. Wait, the user just wants a synthesized response, so maybe a simple ordered list with each album and artist.

Also, check if the order is correct. The SQL used ORDER BY albums.Title, so it's alphabetical by album title. The first one starts with "...", which probably comes before numbers like "20th Century..." since punctuation is considered in sorting. That makes sense.

I should write 

In [29]:
response = query_engine.query('What are some artists? Limit it to 5.')

print(response)

<think>
Okay, I need to answer the user's question: "What are some artists? Limit it to 5." The SQL query they provided selects names from the artists table, orders them alphabetically, and limits the result to five entries. The SQL response gives a list of tuples with the artist names.

First, I should extract the names from each tuple. Each tuple has the structure ('Artist Name',), so I can just take the first element of each. Then, I should present them in a clear, user-friendly format. Since the user asked for some artists, a numbered list would be appropriate. I also need to mention that these are the first five alphabetically since the query uses ORDER BY Name ASC. Maybe I should also note that there are more artists available beyond these five if they need more examples. Let me check the names to ensure there are no duplicates or odd entries. All the names look like valid artists, though some have longer names with orchestras or collaborations. That's fine; the query just pulls 

In [31]:
response = \
    query_engine.query('What are some tracks from the artist AC/DC? Limit it to 3')

print(response)

<think>
Okay, let's tackle this question. The user is asking for some tracks by AC/DC, limited to three. The SQL query provided selects track names by joining the tracks, albums, and artists tables where the artist's name is AC/DC, and limits the result to three entries.

First, I need to make sure that the SQL response has the correct data. The SQL response shows three songs: 'For Those About To Rock (We Salute You)', 'Put The Finger On You', and "Let's Get It Up". These seem like legitimate AC/DC songs. I should verify that these are actual tracks by AC/DC to ensure there's no error in the database. A quick mental check: "For Those About To Rock" is definitely a classic by them. The other two titles also ring a bell as AC/DC songs, though maybe not as well-known as some others.

Next, the user wants the response to be synthesized from the query results. So the task here is to present the information in a natural, conversational way without just listing the SQL output. The answer shou

In [32]:
print(response.metadata['sql_query'])

SELECT tracks.Name FROM tracks INNER JOIN albums ON tracks.AlbumId = albums.AlbumId INNER JOIN artists ON albums.ArtistId = artists.ArtistId WHERE artists.Name = 'AC/DC' LIMIT 3;


## 5. Structured Data Extraction

An important use case for function calling is extracting structured objects. LlamaIndex provides an intuitive interface for this through `structured_predict` - simply define the target Pydantic class (can be nested), and given a prompt, we extract out the desired object.

**NOTE**: Since there's no native function calling support with Llama3 / Ollama, the structured extraction is performed by prompting the LLM + output parsing.

In [33]:
from llama_index.core.prompts import PromptTemplate
from pydantic import BaseModel


class Restaurant(BaseModel):
    """A restaurant with name, city, and cuisine."""

    name: str
    city: str
    cuisine: str

llm = llm_replicate


prompt_tmpl = \
    PromptTemplate('Generate a restaurant in a given city {city_name}')

In [34]:
restaurant_obj = \
    llm.structured_predict(
        Restaurant, 
        prompt_tmpl, 
        city_name='Miami',
    )
print(restaurant_obj)

name='Sabor Cubano Miami' city='Miami' cuisine='Cuban'


## 6. Adding Chat History to RAG (Chat Engine)

In this section we create a stateful chatbot from a RAG pipeline, with our chat engine abstraction.

Unlike a stateless query engine, the chat engine maintains conversation history (through a memory module like buffer memory). It performs retrieval given a condensed question, and feeds the condensed question + context + chat history into the final LLM prompt.

Related resource: https://docs.llamaindex.ai/en/stable/examples/chat_engine/chat_engine_condense_plus_context/

In [35]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import CondensePlusContextChatEngine

memory = ChatMemoryBuffer.from_defaults(token_limit=3900)

chat_engine = \
    CondensePlusContextChatEngine.from_defaults(
        index.as_retriever(),
        memory=memory,
        llm=llm_replicate,
        context_prompt= \
            'You are a chatbot, able to have normal interactions, as well as talk' + \
            ' about the Kendrick and Drake beef.' + \
            'Here are the relevant documents for the context:\n' + \
            '{context_str}' + \
            '\nInstruction: Use the previous chat history, or the context above, to interact and help the user.',
        verbose=True,
    )

In [36]:
response = \
    chat_engine.chat('Tell me about the songs Drake released in the beef.')
print(str(response))

Condensed question: Tell me about the songs Drake released in the beef.
<think>
Okay, the user asked about the songs Drake released in the beef with Kendrick Lamar. Let me go through the provided context to find relevant information.

From page 28, there's a detailed section about Drake's response track "The Heart Part 6" released on May 5. The document mentions that Drake countered Kendrick's "6:16 in LA" by using Kendrick's own "The Heart" series, which is significant because Kendrick's last was "The Heart Part 5." 

The song samples Aretha Franklin's "Let me see you proooove it," which Drake uses to suggest Kendrick's accusations are baseless. It also mentions that Drake directly responds to Kendrick's "Not Like Us" and that the track was written quickly, within 24 hours, showing a reactive approach. Drake's tone here is described as a bit over it but still hinting at escalation.

The assistant needs to structure this information clearly. Start by stating Drake's key track, explain 

In [37]:
response = chat_engine.chat('What about Kendrick?')
print(str(response))

Condensed question: <think>
Okay, so the user initially asked about Drake's songs in the beef with Kendrick Lamar. The assistant provided a detailed answer about Drake's track "The Heart Part 6". Now, the user's follow-up is "What about Kendrick?" The goal is to rephrase this into a standalone question.

First, I need to understand the context. The original question was about Drake, so the follow-up is asking for similar information but about Kendrick. The standalone question needs to make sense without the previous conversation. It should be clear that they're asking about Kendrick's songs in the same beef. 

The user might be wondering which songs Kendrick released during the feud, similar to the information provided about Drake. To make it standalone, the question should explicitly mention the context of the Drake and Kendrick Lamar beef. 

So, the rephrased question would be: "What songs did Kendrick Lamar release in his beef with Drake?" This includes both artists' names and the c

## 7. Agents

Here we build agents with Llama 3. We perform RAG over simple functions as well as the documents above.

### Agents And Tools

In [38]:
import json
from typing import Sequence, List

from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.agent import ReActAgent

import nest_asyncio

nest_asyncio.apply()

### Define Tools

In [39]:
def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two integers and returns the result integer"""
    return a - b


def divide(a: int, b: int) -> int:
    """Divides two integers and returns the result integer"""
    return a / b


multiply_tool = FunctionTool.from_defaults(fn=multiply)
add_tool = FunctionTool.from_defaults(fn=add)
subtract_tool = FunctionTool.from_defaults(fn=subtract)
divide_tool = FunctionTool.from_defaults(fn=divide)

### ReAct Agent

In [40]:
agent = \
    ReActAgent.from_tools(
        [multiply_tool, add_tool, subtract_tool, divide_tool],
        llm=llm_replicate,
        verbose=True,
    )

### Querying

In [41]:
response = agent.chat("What is (121 + 2) * 5?")
print(str(response))

> Running step 5ac42256-f02d-46b9-b99f-b85e90b794ec. Step input: What is (121 + 2) * 5?
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {'a': 121, 'b': 2}
Observation: 123
> Running step bdc8914a-c4d2-4f65-96e2-04b8082ebd12. Step input: None
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: multiply
Action Input: {'a': 123, 'b': 5}
Observation: 615
> Running step baa69ff0-e27d-4c3d-ac42-3130c6ef8dda. Step input: None
Thought: The current language of the user is: English. I can answer without using any more tools. I'll use the user's language to answer
Answer: The result of (121 + 2) * 5 is 615.
The result of (121 + 2) * 5 is 615.


### ReAct Agent With RAG QueryEngine Tools

In [42]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)

from llama_index.core.tools import QueryEngineTool, ToolMetadata

### Create ReAct Agent using RAG QueryEngine Tools

In [43]:
drake_tool = \
    QueryEngineTool(
        drake_index.as_query_engine(),
        metadata= \
            ToolMetadata(
                name='drake_search',
                description="Useful for searching over Drake's life.",
            ),
    )

kendrick_tool = \
    QueryEngineTool(
        kendrick_index.as_query_engine(),
        metadata= \
            ToolMetadata(
                name='kendrick_search',
                description="Useful for searching over Kendrick's life.",
            ),
    )

query_engine_tools = [drake_tool, kendrick_tool]

In [44]:
agent = \
    ReActAgent.from_tools(
        query_engine_tools,  ## TODO: define query tools
        llm=llm_replicate,
        verbose=True,
    )

### Querying

In [45]:
response = agent.chat('Tell me about how Kendrick and Drake grew up')
print(str(response))

> Running step 2b0ca249-3fa6-4f6f-b68f-9db9cf320ec9. Step input: Tell me about how Kendrick and Drake grew up
Thought: The current language of the user is: English. I need to use tools to gather information about both Kendrick and Drake's upbringings.
Action: kendrick_search
Action Input: {'input': 'early life and upbringing'}
Observation: <think>
Okay, let's tackle this query: "early life and upbringing" for Kendrick Lamar. I need to go through the provided context and extract relevant information.

First, looking at the details on page 3. It mentions he attended Centennial High School, had the nickname "Man-Man" due to his precocious behavior. He was a quiet, observant student with a stutter, but excelled academically. His first-grade teacher at Robert E. McNair Elementary encouraged him to write after he used "audacity" correctly. In seventh grade at Vanguard Learning Center, his English teacher Regis Inge introduced him to poetry, connecting it to hip hop. This led him to study rhy